# Time-explicit LCA + datapackages

Once a `TimexLCA` is solved, the time-explicit system is *just matrices fed by `bw_processing` datapackages*. That means you can hand `bw2calc` **extra** datapackages that override individual, time-stamped exchanges — to run **scenarios**, **sensitivity**, and **Monte-Carlo uncertainty** **without rebuilding the model or regenerating databases**.

This notebook assumes you've done the [getting-started](getting_started.ipynb) tutorial. The model below is the standard `electric_vehicle_standalone` example (three background vintages 2020/2030/2040 + a temporally-distributed foreground); we set it up quickly and spend the real attention on the **datapackage layer**.

## Model setup (recap)

Standard time-explicit EV model — clean project, three background vintages with falling CO₂ intensity, a `dynamic` foreground (`ev_production` / `driving` / `used_ev`), temporal distributions on the foreground edges. Nothing new here; collapse if familiar.

In [1]:
import bw2data as bd

bd.projects.set_current("electric_vehicle_standalone")

In [2]:
for db in list(bd.databases):
    del bd.databases[db]

In [3]:
biosphere = bd.Database("biosphere")
biosphere.register()
biosphere.write(
    {
        ("biosphere", "CO2"): {
            "type": "emission",
            "name": "carbon dioxide",
        },
    }
)

background_2020 = bd.Database("background_2020")
background_2020.register()

background_2030 = bd.Database("background_2030")
background_2030.register()

background_2040 = bd.Database("background_2040")
background_2040.register()

background_2020.write({})
background_2030.write({})
background_2040.write({})

background_databases = [
    background_2020,
    background_2030,
    background_2040,
]

100%|██████████| 1/1 [00:00<00:00, 5370.43it/s]

12:34:46+0200 [info     ] Vacuuming database            


In [4]:
process_co2_emissions = {
    "glider": (10, 5, 2.5), # for 2020, 2030 and 2040
    "powertrain": (20, 10, 7.5),
    "battery": (10, 5, 4),
    "electricity": (0.5, 0.25, 0.075),
    "glider_eol": (0.01, 0.0075, 0.005),
    "powertrain_eol": (0.01, 0.0075, 0.005),
    "battery_eol": (1, 0.5, 0.25),
}

node_co2 = biosphere.get("CO2")

for component_name, gwis in process_co2_emissions.items():
    for database, gwi in zip(background_databases, gwis):
        database.new_node(component_name, name=component_name, location="somewhere").save()
        component = database.get(component_name)
        component["reference product"] = component_name        
        component.save()
        production_amount = -1 if "eol" in component_name else 1
        component.new_edge(input=component, amount=production_amount, type="production").save()
        component.new_edge(input=node_co2, amount=gwi, type="biosphere").save()       
    

In [5]:
ELECTRICITY_CONSUMPTION = 0.2 # kWh/km
MILEAGE = 150_000 # km
LIFETIME = 15 # years

# Overall mass: 1200 kg
MASS_GLIDER = 840 # kg
MASS_POWERTRAIN = 80 # kg
MASS_BATTERY = 280 # kg

In [6]:
if "foreground" in bd.databases:
    del bd.databases["foreground"] # to make sure we create the foreground from scratch
foreground = bd.Database("foreground")
foreground.register()

In [7]:
ev_production = foreground.new_node("ev_production", name="production of an electric vehicle", unit="unit")
ev_production['reference product'] = "electric vehicle"
ev_production.save()

driving = foreground.new_node("driving", name="driving an electric vehicle", unit="transport over an ev lifetime")
driving['reference product'] = "transport"
driving.save()

used_ev = foreground.new_node("used_ev", name="used electric vehicle", unit="unit")
used_ev['reference product'] = "used electric vehicle"
used_ev.save()

In [8]:
glider_production = background_2020.get(code="glider")
powertrain_production = background_2020.get(code="powertrain")
battery_production = background_2020.get(code="battery")

ev_production.new_edge(input=ev_production, amount=1, type="production").save()

glider_to_ev = ev_production.new_edge(
    input=glider_production,
    amount=MASS_GLIDER,
    type="technosphere"
)
powertrain_to_ev = ev_production.new_edge(
    input=powertrain_production, 
    amount=MASS_POWERTRAIN, 
    type="technosphere"
)
battery_to_ev = ev_production.new_edge(
    input=battery_production, 
    amount=MASS_BATTERY, 
    type="technosphere"
)

In [9]:
glider_eol = background_2020.get(name="glider_eol")
powertrain_eol = background_2020.get(name="powertrain_eol")
battery_eol = background_2020.get(name="battery_eol")

used_ev.new_edge(input=used_ev, amount=-1, type="production").save()  # -1 as this gets rid of a used car

used_ev_to_glider_eol = used_ev.new_edge(
    input=glider_eol,
    amount=-MASS_GLIDER,
    type="technosphere",
)
used_ev_to_powertrain_eol = used_ev.new_edge(
    input=powertrain_eol,
    amount=-MASS_POWERTRAIN,
    type="technosphere",
)
used_ev_to_battery_eol = used_ev.new_edge(
    input=battery_eol,
    amount=-MASS_BATTERY,
    type="technosphere",
)

In [10]:
electricity_production = background_2020.get(name="electricity")

driving.new_edge(input=driving, amount=1, type="production").save()

driving_to_used_ev = driving.new_edge(input=used_ev, amount=-1, type="technosphere")
ev_to_driving = driving.new_edge(
    input=ev_production, 
    amount=1, 
    type="technosphere"
)
electricity_to_driving = driving.new_edge(
    input=electricity_production,
    amount=ELECTRICITY_CONSUMPTION * MILEAGE,
    type="technosphere",
)

In [11]:
from bw_temporalis import TemporalDistribution, easy_timedelta_distribution
import numpy as np

td_assembly_and_delivery = TemporalDistribution(
    date=np.array([-3, -2], dtype="timedelta64[M]"), amount=np.array([0.2, 0.8])
)

td_glider_production = TemporalDistribution(
    date=np.array([-2, -1, 0], dtype="timedelta64[Y]"), amount=np.array([0.7, 0.1, 0.2])
)

td_produce_powertrain_and_battery = TemporalDistribution(
    date=np.array([-1], dtype="timedelta64[Y]"), amount=np.array([1])
)

td_use_phase = easy_timedelta_distribution(
    start=0,
    end=LIFETIME,
    resolution="Y",
    steps=(LIFETIME + 1),
    kind="uniform", # you can also do "normal" or "triangular" distributions
)

td_disassemble_used_ev = TemporalDistribution(
    date=np.array([LIFETIME + 1], dtype="timedelta64[Y]"), amount=np.array([1])
)

td_treating_waste = TemporalDistribution(
    date=np.array([3], dtype="timedelta64[M]"), amount=np.array([1])
)

In [12]:
glider_to_ev["temporal_distribution"] = td_glider_production
glider_to_ev.save()

powertrain_to_ev["temporal_distribution"] = td_produce_powertrain_and_battery
powertrain_to_ev.save()

battery_to_ev["temporal_distribution"] = td_produce_powertrain_and_battery
battery_to_ev.save()

ev_to_driving["temporal_distribution"] = td_assembly_and_delivery
ev_to_driving.save()

electricity_to_driving["temporal_distribution"] = td_use_phase
electricity_to_driving.save()

driving_to_used_ev["temporal_distribution"] = td_disassemble_used_ev
driving_to_used_ev.save()

used_ev_to_glider_eol["temporal_distribution"] = td_treating_waste
used_ev_to_glider_eol.save()

used_ev_to_powertrain_eol["temporal_distribution"] = td_treating_waste
used_ev_to_powertrain_eol.save()

used_ev_to_battery_eol["temporal_distribution"] = td_treating_waste
used_ev_to_battery_eol.save()

In [13]:
for db in bd.databases:
    bd.Database(db).process()

In [14]:
bd.Method(("GWP", "example")).write(
    [
        (("biosphere", "CO2"), 1),
    ]
)

## Solve the time-explicit LCA

`build_timeline()` + `lci()` as usual. We keep the resulting `TimexLCA` around — its inner `bw2calc` LCA and its `activity_time_mapping` are what the datapackage work below hooks into.

In [15]:
method = ("GWP", "example")

In [16]:
from datetime import datetime

database_dates = {
    "background_2020": datetime.strptime("2020", "%Y"),
    "background_2030": datetime.strptime("2030", "%Y"),
    "background_2040": datetime.strptime("2040", "%Y"),
    "foreground": "dynamic",  # flag databases that should be temporally distributed with "dynamic"
}

In [17]:
from bw_timex import TimexLCA

In [18]:
driving = bd.get_node(database="foreground", code="driving", name="driving an electric vehicle", unit="transport over an ev lifetime")

In [19]:
tlca = TimexLCA({driving: 1}, method, database_dates)

2026-06-22 12:34:47.380 | INFO     | bw_timex.timex_lca:__init__:136 - Initializing TimexLCA object...
2026-06-22 12:34:47.381 | INFO     | bw_timex.timex_lca:__init__:153 - Calculating base LCA...
2026-06-22 12:34:47.397 | INFO     | bw_timex.timex_lca:__init__:170 - Collecting node infos...


In [20]:
tl = tlca.build_timeline(temporal_grouping="month")

2026-06-22 12:34:47.406 | INFO     | bw_timex.timex_lca:build_timeline:328 - No edge filter function provided. Skipping all edges in background databases.
2026-06-22 12:34:47.408 | INFO     | bw_timex.timex_lca:build_timeline:349 - Creating activity time mapping...
2026-06-22 12:34:47.409 | INFO     | bw_timex.timeline_builder:__init__:112 - Traversing supply chain graph...
2026-06-22 12:34:47.423 | INFO     | bw_timex.timeline_builder:build_timeline:186 - Building timeline...
2026-06-22 12:34:47.447 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:630 - Reference date 2040-07-01 00:00:00 is higher than all provided dates. Data will be taken from the closest lower year.


Starting graph traversal
Calculation count: 9


In [21]:
tlca.lci()
tlca.static_lcia()
tlca.static_score

2026-06-22 12:34:47.459 | INFO     | bw_timex.timex_lca:lci:499 - Expanding matrices...
2026-06-22 12:34:47.474 | INFO     | bw_timex.timex_lca:lci:518 - Calculating dynamic inventory...


15261.765039012731

<a id='adding-datapackages'></a>
## Adding datapackages

Everything above ran through a normal `bw2calc` LCA object living **inside** the `TimexLCA`. The next lines are just the explicit form of the `tlca.lci()` / `static_lcia()` shorthands:

In [22]:
import bw2calc as bc

In [23]:
tlca.lca.lci()
tlca.lca.lcia()
tlca.lca.score

15261.765039012731

All the time-explicit data — the expanded technosphere, biosphere and characterization — is stored as **datapackages** on that inner LCA object:

In [24]:
tlca.lca.packages

### Why this matters

Because the solved system is *just matrices fed by datapackages*, we can hand `bw2calc` an **additional** datapackage that overrides specific cells — without touching the inventory or rerunning the timeline. Here we modify the electricity consumed by driving **in a single future year**, then sweep several values at once.

First we need the matrix **coordinates** (row = input, column = consumer) of the exchange we want to override. `activity_time_mapping` maps each `(activity, time)` pair to its matrix id:

In [25]:
tlca.activity_time_mapping

{(('background_2020', 'glider'), 202001): 327395989872181248,
 (('background_2030', 'glider'), 203001): 327395989914124288,
 (('background_2040', 'glider'), 204001): 327395989947678720,
 (('background_2020', 'powertrain'), 202001): 327395989981233152,
 (('background_2030', 'powertrain'), 203001): 327395990014787584,
 (('background_2040', 'powertrain'), 204001): 327395990044147712,
 (('background_2020', 'battery'), 202001): 327395990077702144,
 (('background_2030', 'battery'), 203001): 327395990111256576,
 (('background_2040', 'battery'), 204001): 327395990144811008,
 (('background_2020', 'electricity'), 202001): 327395990178365440,
 (('background_2030', 'electricity'), 203001): 327395990207725568,
 (('background_2040', 'electricity'), 204001): 327395990241280000,
 (('background_2020', 'glider_eol'), 202001): 327395990270640128,
 (('background_2030', 'glider_eol'), 203001): 327395990304194560,
 (('background_2040', 'glider_eol'), 204001): 327395990337748992,
 (('background_2020', 'power

Look up the two ids we need (driving in 2026, the electricity *temporal market* in 2030):

In [26]:
id_driving = tlca.activity_time_mapping[(('temporalized', 'driving'), 202607)]
id_driving

327395990836871182

In [27]:
id_electricity = tlca.activity_time_mapping[(('temporalized', 'electricity'), 203007)]  # the electricity "temporal market" seen by driving in 2030
id_electricity

327395990836871186

### Building the override datapackage

`sequential=True` means the data array is read as a **sequence of scenarios**, one solve per column. We supply three aligned arrays:
- **`indices`** — the `(input, consumer)` coordinate(s) to override.
- **`flip_array`** — `True` because electricity is an *input* (negative in the technosphere).
- **`data`** — one row per coordinate, one column per scenario.

In [28]:
import bw_processing as bwp

In [29]:
datapackage = bwp.create_datapackage(sequential=True) # sequential means: treat data as a sequence of values that are plugged in one after another

In [30]:
import numpy as np

indices = np.array(
    [
        (id_electricity, id_driving),
        # (some_other_input_id, some_other_process_id),  # You can add more exchanges here
    ],
    dtype=bwp.INDICES_DTYPE
)

In [31]:
flip_array = np.array([True])

Scenario values for this exchange. Original amount is `1875` (= 0.2 kWh/km × 150 000 km spread over the use phase), kept as the baseline; the rest model progressively less driving electricity in that year:

In [32]:
data = np.array(
    [
        [1875, 1000, 500, 0],
        # [values for some_other_input if added],
    ]
)

In [33]:
scenario_labels = ["baseline", "reduction", "heavy reduction", "zero"]

Register the array in the datapackage:

In [34]:
datapackage.add_persistent_array(
    matrix="technosphere_matrix",
    indices_array=indices,
    data_array=data,
    flip_array=flip_array,
)

### Solve every scenario

Create a fresh LCA from **the original Timex datapackages plus our override**, with `use_arrays=True` so the sequential array is actually iterated. `keep_first_iteration()` holds the baseline column, then iterating the LCA steps through the remaining scenarios — each one a cheap refactorized solve, no rebuild:

In [35]:
lca = bc.LCA(tlca.fu, method, data_objs=tlca.lca.packages + [datapackage], use_arrays=True)

In [36]:
lca.lci()
lca.lcia()

In [37]:
lca.keep_first_iteration() # keep the base values

for _, label in zip(lca, scenario_labels):
    print(label, lca.score)

baseline 15261.765039012731
reduction 15050.671288882346
heavy reduction 14930.046288807838
zero 14809.421288733332


## Uncertainty: Monte Carlo on a time-stamped exchange

The same coordinate trick drives **uncertainty propagation**. Instead of a fixed sequence of values, we attach a *distribution* to the exchange via `add_persistent_vector` with a `distributions_array`, then let `bw2calc` resample it with `use_distributions=True`.

Here we say the 2030 driving-electricity amount is **Normal(mean=1875, sd=250)** and propagate it. Only this one cell carries uncertainty — every other Timex datapackage stays at its static value — so we isolate *how much that single time-stamped input moves the result*.

In [38]:
import stats_arrays as sa

unc_dp = bwp.create_datapackage()

distributions = np.array(
    [
        # (uncertainty_type, loc, scale, shape, minimum, maximum, negative)
        (sa.NormalUncertainty.id, 1875.0, 250.0, np.nan, np.nan, np.nan, False),
    ],
    dtype=bwp.UNCERTAINTY_DTYPE,
)

unc_dp.add_persistent_vector(
    matrix="technosphere_matrix",
    indices_array=indices,            # reuse the coordinate from the scenario sweep
    data_array=np.array([1875.0]),    # static fallback (used if distributions are off)
    flip_array=flip_array,
    distributions_array=distributions,
)

Run it with `use_distributions=True` and pull a few hundred samples. `next(lca)` resamples every uncertain datapackage and refactorizes; we recompute the score each draw:

In [39]:
lca_mc = bc.LCA(
    tlca.fu,
    method,
    data_objs=tlca.lca.packages + [unc_dp],
    use_distributions=True,
)
lca_mc.lci()
lca_mc.lcia()

mc_scores = []
for _ in range(250):
    next(lca_mc)
    lca_mc.lcia()
    mc_scores.append(lca_mc.score)

mc_scores = np.array(mc_scores)
print(f"mean  {mc_scores.mean():.1f}")
print(f"std   {mc_scores.std():.1f}")
print(f"5-95% {np.percentile(mc_scores, 5):.1f} - {np.percentile(mc_scores, 95):.1f}")

mean  15260.4
std   58.8
5-95% 15169.4 - 15359.2


The spread here reflects uncertainty in **one exchange at one point in time**. This is the smallest useful Monte Carlo datapackage: one matrix coordinate, one fallback value, one flip flag, and one distribution row.

### Extending uncertainty to further electricity inputs

To propagate uncertainty across more of the time-explicit matrix, add one row per additional matrix coordinate. Here we keep the single-exchange example above intact, then build a second datapackage that puts uncertainty on **all yearly electricity inputs to driving**. The mean stays at the time-explicit electricity amount for each year, while the relative standard deviation grows for exchanges further into the future.

In [40]:
unc_dp_all_electricity = bwp.create_datapackage()

electricity_exchanges = sorted(
    (
        (time, matrix_id)
        for ((database, name), time), matrix_id in tlca.activity_time_mapping.items()
        if database == "temporalized" and name == "electricity"
    ),
    key=lambda item: item[0],
)

electricity_indices = np.array(
    [(matrix_id, id_driving) for _, matrix_id in electricity_exchanges],
    dtype=bwp.INDICES_DTYPE,
)

electricity_amounts = ELECTRICITY_CONSUMPTION * MILEAGE * td_use_phase.amount
assert len(electricity_indices) == len(electricity_amounts)

electricity_years = np.array([time // 100 for time, _ in electricity_exchanges])
years_from_start = electricity_years - electricity_years.min()
relative_sd = 0.10 + 0.025 * years_from_start
electricity_sd = electricity_amounts * relative_sd

all_electricity_distributions = np.array(
    [
        # (uncertainty_type, loc, scale, shape, minimum, maximum, negative)
        (sa.NormalUncertainty.id, float(mean), float(sd), np.nan, 0.0, np.nan, False)
        for mean, sd in zip(electricity_amounts, electricity_sd)
    ],
    dtype=bwp.UNCERTAINTY_DTYPE,
)

unc_dp_all_electricity.add_persistent_vector(
    matrix="technosphere_matrix",
    indices_array=electricity_indices,
    data_array=electricity_amounts,  # static fallback (used if distributions are off)
    flip_array=np.ones(len(electricity_indices), dtype=bool),
    distributions_array=all_electricity_distributions,
)

for time, mean, sd, rel in zip(
    electricity_years, electricity_amounts, electricity_sd, relative_sd
):
    print(f"{time}: mean={mean:.1f}, sd={sd:.1f} ({rel:.1%})")

2026: mean=1875.0, sd=187.5 (10.0%)
2027: mean=1875.0, sd=234.4 (12.5%)
2028: mean=1875.0, sd=281.3 (15.0%)
2029: mean=1875.0, sd=328.1 (17.5%)
2030: mean=1875.0, sd=375.0 (20.0%)
2031: mean=1875.0, sd=421.9 (22.5%)
2032: mean=1875.0, sd=468.8 (25.0%)
2033: mean=1875.0, sd=515.6 (27.5%)
2034: mean=1875.0, sd=562.5 (30.0%)
2035: mean=1875.0, sd=609.4 (32.5%)
2036: mean=1875.0, sd=656.2 (35.0%)
2037: mean=1875.0, sd=703.1 (37.5%)
2038: mean=1875.0, sd=750.0 (40.0%)
2039: mean=1875.0, sd=796.9 (42.5%)
2040: mean=1875.0, sd=843.8 (45.0%)
2041: mean=1875.0, sd=890.6 (47.5%)


Run the same Monte Carlo loop with the expanded datapackage. The wider output spread now reflects uncertainty in all modeled use-phase electricity inputs, with later exchanges sampled more widely than earlier ones.

In [41]:
lca_mc_all_electricity = bc.LCA(
    tlca.fu,
    method,
    data_objs=tlca.lca.packages + [unc_dp_all_electricity],
    use_distributions=True,
)
lca_mc_all_electricity.lci()
lca_mc_all_electricity.lcia()

all_electricity_mc_scores = []
for _ in range(250):
    next(lca_mc_all_electricity)
    lca_mc_all_electricity.lcia()
    all_electricity_mc_scores.append(lca_mc_all_electricity.score)

all_electricity_mc_scores = np.array(all_electricity_mc_scores)
print(f"mean  {all_electricity_mc_scores.mean():.1f}")
print(f"std   {all_electricity_mc_scores.std():.1f}")
print(
    f"5-95% {np.percentile(all_electricity_mc_scores, 5):.1f} - "
    f"{np.percentile(all_electricity_mc_scores, 95):.1f}"
)

mean  15260.4
std   324.8
5-95% 14718.3 - 15737.5
